Exploration,Data_preprocessing

In [ ]:
df = pd.read_csv('transaction_dataset.csv')
pd.set_option('display.max_columns', None)
df.head()
df.drop(['Unnamed: 0','Index','Address'],axis =1,inplace = True)
df.head()
print("Count of each label:\n", df['FLAG'].value_counts())

print("Number of rows with undetermined fraud status:", df['FLAG'].isnull().sum())
df.shape
# Important information about each feature
df.describe()

print("Number of null values per feature:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print()

print("Number of unique values in ERC20_most_rec_token_type:", df[' ERC20_most_rec_token_type'].nunique())
print("Number of unique values in ERC20 most sent token type:", df[' ERC20 most sent token type'].nunique())
# Find duplicate rows
duplicate_rows = df[df.duplicated()]
duplicate_rows
# Drop duplicate rows
df.drop_duplicates(inplace=True)

# Rename feature columns to remove leading/trailing spaces
df_copy = df.copy()
df_copy.columns = df_copy.columns.str.strip()

# Fill numeric features with mean
for col in df_copy.columns:
    if df_copy[col].isnull().sum() == 829:
        df_copy[col].fillna(df_copy[col].mean(), inplace=True)

# Print columns that still have null values
print("Columns still containing null values:")
print(df_copy.isnull().sum()[df_copy.isnull().sum() > 0])
print()

# View value counts of categorical feature
categorical_feature = 'ERC20_most_rec_token_type'
print(f"Value counts of feature {categorical_feature}:\n{df_copy[categorical_feature].value_counts()}")
# Clean categorical features - replace 0 with null since 0 has no real meaning in categorical features
categorical_features = ['ERC20_most_rec_token_type', 'ERC20 most sent token type']
df_copy[categorical_features] = df_copy[categorical_features].replace({'0': np.nan})

# Print processed features
print("Processed features:")
print(df_copy[categorical_features])
# View null ratio of a specific feature
null_percentage = df_copy['ERC20 most sent token type'].isnull().sum() / len(df_copy)
print("Null ratio of ERC20 most sent token type:", null_percentage)
# Create a list of features with zero variance
to_drop = list(df_copy.var(numeric_only=True)[df_copy.var(numeric_only=True) == 0].keys())
print("List of zero-variance features:", to_drop)

# Drop zero-variance features
df_copy.drop(to_drop, inplace=True, axis=1)
print("Data description after dropping zero-variance features:")
print(df_copy.describe())

# Find features with near-zero standard deviation
low_std_features = list(df_copy.std(numeric_only=True)[df_copy.std(numeric_only=True) < 0.001].keys())
print("List of features with near-zero standard deviation:", low_std_features)
df_copy.drop(['min value sent to contract','max val sent to contract','avg value sent to contract'\
              ,'total ether sent contracts']\
             ,axis =1 , inplace = True)
# Fill missing values in categorical features using mode
df_copy = df_copy.fillna(df_copy.mode().iloc[0])

# Print columns that still have null values
remaining_nulls = df_copy.isnull().sum()[df_copy.isnull().sum() > 0]
print("Columns still containing null values:")
print(remaining_nulls)
df_copy[['ERC20_most_rec_token_type', 'ERC20 most sent token type']].columns
# Handle features with more than 50 unique categories
# We use Label Encoding for features with more than 50 categories

# Create a copy for processing
df_encoded = df_copy.copy()

# Store features that need Label Encoding
label_encoding_features = []

# Store categorical features and their encoding dictionaries
categorical_features_dict = {}

# Iterate through the features to process
for feature in df_copy[['ERC20_most_rec_token_type', 'ERC20 most sent token type']].columns:
    if df_copy[feature].nunique() >= 50:  # Use Label Encoding if more than 50 unique categories
        label_encoding_features.append(feature)
        categorical_features_dict[feature] = {}  # Dictionary maps original value to encoded integer
        i = 1  # Running index (category)
        
        # Map feature values to category integers using dictionary
        for sample in df_copy[feature]:
            if sample not in categorical_features_dict[feature].keys() and sample is not np.nan:  # Replace each value (not null)
                categorical_features_dict[feature][sample] = i
                i += 1

        # Apply Label Encoding to both the copy and original data
        df_encoded[feature].replace(categorical_features_dict[feature], inplace=True)

# Print Label Encoding features and their encoding dictionaries
print("Features requiring Label Encoding:", label_encoding_features)
print("Encoding dictionaries for categorical features:", categorical_features_dict)
# Some outliers were found in the scatter plots plotted earlier
# Next, we check whether removing these outliers improves model performance
outliers_indices = df_copy[(df_copy['Received Tnx'] > 3000) & (df_copy['FLAG'] == 1) & (df_copy['Received Tnx'] < 4000)].index

# Print indices of detected outliers
print("Indices of detected outliers:")
print(outliers_indices)
# Set plot style
sns.set(style='darkgrid')

# Create a function to plot boxplot and stripplot
def plot_boxplot_and_stripplot(data, title):
    plt.subplots(figsize=(14, 8))
    boxplot = sns.boxplot(data=data)
    stripplot = sns.stripplot(data=data, marker="o", alpha=0.3, color="blue")
    boxplot.axes.set_title(title, fontsize=16)
    boxplot.set_xlabel("Conditions", fontsize=14)
    boxplot.set_ylabel("Values", fontsize=14)
    plt.show()

# Plot boxplot and stripplot for 'Avg min between received tnx'
data1 = df_copy['Avg min between received tnx'][df_copy['FLAG'] == 0]
data2 = df_copy['Avg min between received tnx'][df_copy['FLAG'] == 1]
plot_boxplot_and_stripplot(data=[data1, data2], title="Distribution of Avg min between received tnx")

# Plot boxplot and stripplot for 'Time Diff between first and last (Mins)'
data1 = df_copy['Time Diff between first and last (Mins)'][df_copy['FLAG'] == 0]
data2 = df_copy['Time Diff between first and last (Mins)'][df_copy['FLAG'] == 1]
plot_boxplot_and_stripplot(data=[data1, data2], title="Distribution of Time Diff between first and last (Mins)")

# Plot boxplot and stripplot for 'Avg min between received tnx'
data1 = df_copy['Avg min between received tnx'][df_copy['FLAG'] == 0]
data2 = df_copy['Avg min between received tnx'][df_copy['FLAG'] == 1]
plot_boxplot_and_stripplot(data=[data1, data2], title="Distribution of Avg min between received tnx")
# Detect and handle outliers
# Remove certain outliers; adjust as needed

# Find and drop outliers in 'Time Diff between first and last (Mins)'
outliers_1 = df_copy[
    (df_copy['FLAG'] == 1) & 
    (df_copy['Time Diff between first and last (Mins)'] > 600000)
]
df_copy.drop(outliers_1.index, axis=0, inplace=True)

# Find and drop outliers in 'Avg min between received tnx'
outliers_2 = df_copy[
    (df_copy['FLAG'] == 1) & 
    (df_copy['Avg min between received tnx'] > 60000)
]
df_copy.drop(outliers_2.index, axis=0, inplace=True)

# Find outliers in 'Received Tnx'
outliers_3 = df_copy[
    (df_copy['FLAG'] == 1) & 
    (df_copy['Received Tnx'] > 8000)
]

# Print detected outliers
print("Detected outliers 1:")
print(outliers_1)
print()

print("Detected outliers 3:")
print(outliers_3)

# Print outlier indices in 'Avg min between received tnx' (non-fraud)
print("Outlier indices in Avg min between received tnx (non-fraud):")
print(df_copy['Avg min between received tnx'][
    (df_copy['FLAG'] == 0) & 
    (df_copy['Avg min between received tnx'] > 400000)
].index)
print()

# Print outlier indices in 'Avg min between received tnx' (fraud)
print("Outlier indices in Avg min between received tnx (fraud):")
print(df_copy['Avg min between received tnx'][
    (df_copy['FLAG'] == 1) & 
    (df_copy['Avg min between received tnx'] > 140000)
].index)
# Calculate and handle outliers for each feature
for feature in df_copy.drop('FLAG', axis=1).columns:
    # Ensure the feature is numeric
    if np.issubdtype(df_copy[feature].dtype, np.number):
        IQR = np.percentile(df_copy[feature], 75) - np.percentile(df_copy[feature], 25)
        lower_limit = np.percentile(df_copy[feature], 25) - 1.5 * IQR
        upper_limit = np.percentile(df_copy[feature], 75) + 1.5 * IQR
        
        # Count the number of outliers
        outliers_count = ((df_copy[feature] > upper_limit) | (df_copy[feature] < lower_limit)).sum()
        
        # Calculate the proportion of outliers
        outlier_proportion = outliers_count / df_copy.shape[0]
        
        # Set the maximum acceptable outlier proportion threshold
        max_outlier_proportion = 0.07
        
        if outlier_proportion <= max_outlier_proportion:
            print(f"Feature '{feature}' has an outlier proportion of {outlier_proportion:.2%} and will be considered.")
#             Optionally remove outliers from the dataset here
            df_copy = df_copy[~((df_copy[feature] > upper_limit) | (df_copy[feature] < lower_limit))]